# CountVideoBench — Analysis Notebook

Reproduces the per-model evaluation on CountVideoBench from the paper:

1. Loading the per-model CSV files (`../paper_results/`).
2. Per-model summary metrics (Table 2) and Exact Match by object class (Table 6).
3. Coverage by cardinality (Figure 3).
4. Exact Match heatmap (Figure 4).
5. Per-model prediction-bias histograms (Figure 5 and appendix).
6. Model rows of the human-study subset (Table 4, Figure 7).

The computations live in `config.py`, `data.py`, `metrics.py` and `plots.py`;
this notebook only wires them together. `compute_metrics.py` does the same
without Jupyter and writes every table and figure to `output/`.

## Setup

In [ ]:
from pathlib import Path

import pandas as pd

from config import (
    DEFAULT_INPUT_PATTERN,
    DEFAULT_OUTPUT_DIR,
    BIAS_FIGURE_SUBDIR,
    BIAS_TABLE_SUBDIR,
    HUMAN_SUBSET_PATH,
    benchmark_cardinalities,
    apply_matplotlib_style,
    build_model_palette,
)
from data import load_model_csvs, preprocess, normalise_model_list, palette_order, select_videos
from metrics import (
    coverage_by_cardinality,
    exact_match_by_cardinality,
    summary_metrics,
    local_irregularities,
    cardinality_report,
    exact_match_by_visual_attributes,
)
from plots import (
    plot_line_by_cardinality,
    plot_accuracy_heatmap,
    plot_bias_barplot,
)

apply_matplotlib_style()

OUTPUT_DIR = Path(DEFAULT_OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load data

One CSV per model, as in `../paper_results/` (counts used in the paper).
Model names are mapped to the names displayed in the paper. Empty answers
(no count) are kept and handled by each metric as abstentions.

In [ ]:
df_raw, raw_models = load_model_csvs(DEFAULT_INPUT_PATTERN)
df = preprocess(df_raw)
models = normalise_model_list(raw_models)
palette = build_model_palette(palette_order(models))

print(f"Loaded {len(df):,} rows across {len(models)} models.")
print("Models:", models)

## 2. Per-model summary metrics (Table 2)

`EM`, `Acc@10%`, `MAE`, `RMSE`, `Coverage`, `Support size`, `Entropy` and
`Perplexity` for every model, plus the `Oracle` reference row (predictions =
ground truth on every video). Perplexity counts abstentions as an extra
category, as in the paper.

In [ ]:
metrics = summary_metrics(df, include_oracle=True)
metrics.to_csv(OUTPUT_DIR / "models_summary_metrics.csv")
metrics.round(2)

Largest local excess and deficit of each model's prediction histogram (Section 6.3).

In [ ]:
irregularities = local_irregularities(df)
irregularities.to_csv(OUTPUT_DIR / "local_irregularities.csv")
irregularities

### Exact Match by object class (Table 6)

In [ ]:
attribute_em_tables = exact_match_by_visual_attributes(df)
for name, table in attribute_em_tables.items():
    table.to_csv(OUTPUT_DIR / f"exact_match_by_{name}.csv")

print("Exact Match by shape")
display(attribute_em_tables["shape"].style.format("{:.1%}"))

print("Exact Match by colour")
display(attribute_em_tables["color"].style.format("{:.1%}"))

## 3. Coverage by cardinality (Figure 3)

Proportion of videos for which a model produces a numerical answer,
as a function of the ground-truth count.

In [ ]:
coverage_long = coverage_by_cardinality(df)
coverage_by_cardinality(df, pivot=True).to_csv(OUTPUT_DIR / "coverage_evolution.csv")

plot_line_by_cardinality(
    coverage_long,
    y="coverage", y_label="Coverage",
    palette=palette, percent=True,
    save_to=OUTPUT_DIR / "coverage_curve.pdf",
)

## 4. Exact Match heatmap (Figure 4)

Per-cardinality Exact Match for each model. Non-numerical answers are
counted as incorrect (consistent with Eq. 1).

In [ ]:
em_long = exact_match_by_cardinality(df)
exact_match_by_cardinality(df, pivot=True).to_csv(OUTPUT_DIR / "heatmap_accuracy_overall.csv")

plot_accuracy_heatmap(
    em_long, value="exact_match",
    cmap="Blues", annotate=True,
    save_to=OUTPUT_DIR / "heatmap_accuracy_overall.pdf",
)

## 5. Prediction-bias histograms (Figure 5 and appendix)

One stacked histogram per model showing the distribution of predicted
counts, split into correct (green) and incorrect (red) predictions.
Integer values outside the benchmark grid are shaded grey. The dashed line
marks 100, the number of videos per cardinality. One table per model is
also written to `output/bias_tables/`.

In [ ]:
in_grid = benchmark_cardinalities()
bias_tab_dir = OUTPUT_DIR / BIAS_TABLE_SUBDIR
bias_tab_dir.mkdir(parents=True, exist_ok=True)

for model in models:
    print(model)
    safe_name = model.replace("/", "_")
    cardinality_report(df, model).to_csv(bias_tab_dir / f"{safe_name}.csv", index=False)

    plot_bias_barplot(
        df, model,
        in_dataset_cardinalities=in_grid,
        save_to=OUTPUT_DIR / BIAS_FIGURE_SUBDIR / f"{safe_name}_bias_barplot.pdf",
    )

## 6. Human-study subset (Table 4 and Figure 7, model rows)

The same metrics on the 104 videos of the human study (Appendix C). The
human rows of Table 4 and Figure 7 come from the participants' answers,
which are not part of this folder.

In [ ]:
df_subset = select_videos(df, pd.read_csv(HUMAN_SUBSET_PATH)["video_id"])
subset_dir = OUTPUT_DIR / "human_subset"
subset_dir.mkdir(parents=True, exist_ok=True)

subset_metrics = summary_metrics(df_subset, include_oracle=False)
subset_metrics.to_csv(subset_dir / "models_summary_metrics.csv")
exact_match_by_cardinality(df_subset, pivot=True).to_csv(subset_dir / "heatmap_accuracy_overall.csv")
subset_metrics[["EM", "Acc@10%", "Coverage", "Support size", "Perplexity"]].round(1)